# Phase 2 — Annotation

This phase samples passages from the preprocessed corpus for manual annotation. Sampled passages are exported to a CSV for labeling, then split into train/test/ambiguous sets for downstream modeling.

### Labels
- **0 — Substantive**: specific, measurable, third-party verified claims
- **1 — Greenwashing-risk**: vague, unverified, or contradicted by auditor ratings
- **2 — Ambiguous**: genuine uncertainty; excluded from modeling

### Pipeline Overview
**`passages.jsonl` → stratified sample → manual annotation → `labeled_passages.jsonl` + `test_set.jsonl`**

In [32]:
import json
import random
from pathlib import Path
from textwrap import wrap

import pandas as pd
from sklearn.metrics import cohen_kappa_score
from codecarbon import EmissionsTracker

In [33]:
ROOT        = Path('..').resolve()
PASSAGES    = ROOT / 'data' / 'extracted' / 'passages.jsonl'
LABELED_DIR = ROOT / 'data' / 'labeled'
LABELED_DIR.mkdir(parents=True, exist_ok=True)

CARBON_DIR = ROOT / 'results' / 'carbon'
CARBON_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_PER_BRAND = 20   # 20 × 16 brands = ~320 total
TEST_SET_N       = 48   # ~15% held-out test set
RANDOM_SEED      = 42

In [34]:
tracker = EmissionsTracker(
    project_name="green-claims-nlp",
    output_dir=str(CARBON_DIR),
    output_file="emissions.csv",
    log_level="error",
)
tracker.start()
print("Carbon tracking started.")

Carbon tracking started.


## Step 2 — Sample passages (stratified by brand)

Loads all passages from passages.jsonl, groups them by brand, then randomly picks 20 from each. The stratification matters — without it, brands with more passages (Puma: 782, H&M: 564) would dominate the labeled set, and small brands (Eileen Fisher: 26) might get nothing. 

Each sampled passage gets an `annotation_id` for tracking.                                                       

In [35]:
with open(PASSAGES) as f:
    all_passages = [json.loads(line) for line in f]

# Stratified sample: ~17 per brand
random.seed(RANDOM_SEED)
by_brand: dict[str, list] = {}
for p in all_passages:
    by_brand.setdefault(p['brand'], []).append(p)

sample = []
for brand, ps in by_brand.items():
    n = min(SAMPLE_PER_BRAND, len(ps))
    sampled = random.sample(ps, n)
    sample.extend(sampled)
    print(f"{brand:12s}: {n} sampled from {len(ps):,} available")

print(f"\nTotal sample: {len(sample)} passages")

# Assign annotation IDs
for i, p in enumerate(sample):
    p['annotation_id'] = i

H&M         : 20 sampled from 553 available
Zara        : 20 sampled from 524 available
Shein       : 20 sampled from 307 available
Kering      : 20 sampled from 89 available
Lululemon   : 20 sampled from 235 available
Ralph Lauren: 20 sampled from 172 available
Puma        : 20 sampled from 758 available
Everlane    : 20 sampled from 198 available
Patagonia   : 20 sampled from 448 available
Eileen Fisher: 20 sampled from 26 available
Reformation : 20 sampled from 171 available
L'Oreal     : 20 sampled from 23 available
Unilever    : 20 sampled from 492 available
Sephora     : 20 sampled from 139 available
Lush        : 20 sampled from 71 available
L'Occitane  : 20 sampled from 279 available

Total sample: 320 passages


## Step 3 — Export to CSV / Load labels back     

**First Run (No Labels Yet)**: 
- check if `to_annotate.csv` exists and covers all current brands 
- if not, generates a fresh csb with columns
- if brands changed but file already existed, preserves any labels already filled in by matching on `annotation_id` 
- prints how many passages still need labels 

**If labels already filled:** 
- reads `to_annotate.csv` back, skips rows where label is blank or invalid 
- builds a `labeled_so_far` dictionary keyed by `annotation_id` 

In [37]:
ANNOTATE_CSV = LABELED_DIR / 'to_annotate.csv'

# Check if an existing CSV covers the same brands — regenerate if brands have changed
existing_brands: set[str] = set()
if ANNOTATE_CSV.exists():
    existing_brands = set(pd.read_csv(ANNOTATE_CSV)['brand'].unique())

sampled_brands = {p['brand'] for p in sample}
needs_regen = not ANNOTATE_CSV.exists() or not sampled_brands.issubset(existing_brands)

if needs_regen:
    export_df = pd.DataFrame([{
        'annotation_id': p['annotation_id'],
        'brand':         p['brand'],
        'industry':      p['industry'],
        'role':          p['role'],
        'text':          p['text'],
        'label':         '',    # fill in: 0 / 1 / 2
        'notes':         '',
    } for p in sample])

    # Preserve any existing labels if regenerating for new brands
    if ANNOTATE_CSV.exists():
        old_df = pd.read_csv(ANNOTATE_CSV)
        labeled_old = old_df[old_df['label'].notna() & (old_df['label'].astype(str).str.strip() != '')]
        old_labels = dict(zip(labeled_old['annotation_id'], labeled_old['label']))
        old_notes  = dict(zip(labeled_old['annotation_id'], labeled_old['notes']))
        export_df['label'] = export_df['annotation_id'].map(old_labels).fillna('')
        export_df['notes'] = export_df['annotation_id'].map(old_notes).fillna('')
        print(f"Preserved {len(old_labels)} existing labels from previous run.")

    export_df.to_csv(ANNOTATE_CSV, index=False)
    print(f"Exported {len(export_df)} passages → {ANNOTATE_CSV}")
    print("\nNext step: open to_annotate.csv, fill in the 'label' column, save, then re-run from Step 4.")
else:
    print(f"to_annotate.csv already exists and covers all brands — proceeding to load labels.")

# Load annotations if the CSV has labels filled in
labeled_so_far: dict[int, dict] = {}
if ANNOTATE_CSV.exists():
    ann_df = pd.read_csv(ANNOTATE_CSV)
    filled = ann_df[ann_df['label'].notna() & (ann_df['label'].astype(str).str.strip() != '')]
    for _, row in filled.iterrows():
        try:
            label = int(row['label'])
        except (ValueError, TypeError):
            continue
        rec = {
            'annotation_id': int(row['annotation_id']),
            'brand':         row['brand'],
            'industry':      row['industry'],
            'role':          row['role'],
            'text':          row['text'],
            'label':         label,
            'notes':         str(row['notes']) if pd.notna(row['notes']) else '',
        }
        labeled_so_far[rec['annotation_id']] = rec

print(f"\nLabeled so far: {len(labeled_so_far)} / {len(sample)} passages")
if len(labeled_so_far) < len(sample):
    unlabeled = len(sample) - len(labeled_so_far)
    print(f"  {unlabeled} passages still need labels. Fill in to_annotate.csv and re-run from Step 4.")

to_annotate.csv already exists and covers all brands — proceeding to load labels.

Labeled so far: 320 / 320 passages


## Step 4 — Split and save labeled sets

1. separates the three label classes
2. propotional rain/test split: the test set is strategied by label then calculates how many 0s and 1s to pull proportionally so the test set reflects the same class balance as the full labeled set. This prevents the test set from being accidentally all-os. 
3. saves three files: `labeled_passages.jsonl`, `test_set.jsonl`, and `ambiguous_passages.jsonl`

In [38]:
if not labeled_so_far:
    print("No labels found. Fill in to_annotate.csv first.")
else:
    all_labeled = list(labeled_so_far.values())
    ambiguous   = [p for p in all_labeled if p['label'] == 2]
    binary      = [p for p in all_labeled if p['label'] in (0, 1)]

    print(f"Label 0 (substantive):    {sum(1 for p in binary if p['label'] == 0)}")
    print(f"Label 1 (greenwash-risk): {sum(1 for p in binary if p['label'] == 1)}")
    print(f"Label 2 (ambiguous):      {len(ambiguous)}")

    random.seed(RANDOM_SEED)
    label0 = [p for p in binary if p['label'] == 0]
    label1 = [p for p in binary if p['label'] == 1]

    n0 = round(TEST_SET_N * len(label0) / max(len(binary), 1))
    n1 = TEST_SET_N - n0
    test_set  = random.sample(label0, min(n0, len(label0))) + random.sample(label1, min(n1, len(label1)))
    test_ids  = {p['annotation_id'] for p in test_set}
    train_set = [p for p in binary if p['annotation_id'] not in test_ids]

    print(f"\nTrain: {len(train_set)} | Test: {len(test_set)} | Ambiguous (excluded): {len(ambiguous)}")

    def save_jsonl(records, path):
        with open(path, 'w') as f:
            for r in records:
                f.write(json.dumps(r) + '\n')

    save_jsonl(train_set, LABELED_DIR / 'labeled_passages.jsonl')
    save_jsonl(test_set,  LABELED_DIR / 'test_set.jsonl')
    save_jsonl(ambiguous, LABELED_DIR / 'ambiguous_passages.jsonl')

    print("\nSaved:")
    print(f"  {LABELED_DIR / 'labeled_passages.jsonl'}")
    print(f"  {LABELED_DIR / 'test_set.jsonl'}")
    print(f"  {LABELED_DIR / 'ambiguous_passages.jsonl'}")
    print("\n⚠  Test set is now locked. Do not re-examine or adjust based on model results.")

Label 0 (substantive):    98
Label 1 (greenwash-risk): 42
Label 2 (ambiguous):      180

Train: 92 | Test: 48 | Ambiguous (excluded): 180

Saved:
  /Users/mandy.sun/green-claims-nlp/data/labeled/labeled_passages.jsonl
  /Users/mandy.sun/green-claims-nlp/data/labeled/test_set.jsonl
  /Users/mandy.sun/green-claims-nlp/data/labeled/ambiguous_passages.jsonl

⚠  Test set is now locked. Do not re-examine or adjust based on model results.


In [39]:
emissions = tracker.stop()
print(f"Total annotation emissions: {emissions:.4f} kg CO2e")
print(f"Saved to: {CARBON_DIR / 'annotation_emissions.csv'}")

Total annotation emissions: 0.0005 kg CO2e
Saved to: /Users/mandy.sun/green-claims-nlp/results/carbon/annotation_emissions.csv
